# Time Series Final Project — Code (Problems 1–3)

In [ ]:
import os, json, warnings, time, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy import stats as scistats
from IPython.display import Markdown, display

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

RNG_SEED = 123
LOG2PI = math.log(2.0 * math.pi)

## Problem 1. ARIMA model and Kalman Filter

For this problem, use the dataset `data/alcoa_volatility.txt`. The dataset consists the realized daily
volatility series of Alcoa stock returns from January 2, 2003 to May 7, 2004. The volatility series
is constructed using 20-minute intra-daily log returns.

**(a)** Estimate the local trend model in Equations (11.1) and (11.2) in the slide Week 11.
Write down the exact equation of your estimated model.

In [ ]:
# Load data (single column, 340 obs; no dates -> use index t = 1..340).
y_values = pd.read_csv("data/alcoa_volatility.txt", header=None, names=["volatility"], sep=r"\s+")
y = pd.Series(y_values["volatility"].to_numpy(), index=np.arange(1, len(y_values) + 1),
             name="Realized volatility")
print(f"Number of observations: {len(y)}  (t = {y.index[0]}..{y.index[-1]})")

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(y.index, y, color="black", linewidth=1.1, label="Observed realized volatility")
ax.set_title("Observed Realized Daily Volatility of Alcoa Stock")
ax.set_xlabel("Observation index t"); ax.set_ylabel("Realized volatility"); ax.legend()
plt.show()

# Local level model (Eq. 11.1-11.2): y_t = mu_t + e_t ; mu_{t+1} = mu_t + eta_t.
# Estimated by maximum likelihood through the Kalman filter.
model_ll = sm.tsa.UnobservedComponents(y, level="local level")
res_ll = model_ll.fit(disp=False)

sigma2_e   = float(res_ll.params["sigma2.irregular"])   # observation-noise variance
sigma2_eta = float(res_ll.params["sigma2.level"])        # level-innovation variance
q = sigma2_eta / sigma2_e
theta_ma = (np.sqrt(q**2 + 4*q) - 2 - q) / 2.0           # implied IMA(1,1) MA coefficient
sigma2_eps = -sigma2_e / theta_ma

print(res_ll.summary())
print(f"\nsigma_e^2 (irregular) = {sigma2_e:.6f}")
print(f"sigma_eta^2 (level)   = {sigma2_eta:.6f}")
print(f"signal-to-noise q     = {q:.6f}  -> implied ARIMA(0,1,1): theta = {theta_ma:.4f}, "
      f"innovation var = {sigma2_eps:.4f}")

display(Markdown(
    "**Estimated model (parameters from `result.params`):**\n\n"
    rf"$$y_t=\mu_t+e_t,\ e_t\sim N(0,\,{sigma2_e:.6f}),\qquad "
    rf"\mu_{{t+1}}=\mu_t+\eta_t,\ \eta_t\sim N(0,\,{sigma2_eta:.6f}).$$"
    "\n\nReduced form: ARIMA(0,1,1), "
    rf"$\Delta y_t=\varepsilon_t{theta_ma:.4f}\varepsilon_{{t-1}}$ "
    "(local level model is estimable by the Kalman filter and is equivalent to an IMA(1,1))."
))

**(b)** Obtain time plots for the filtered variables with pointwise 95% confidence interval.
What is the mathematical expression of these "filtered variables"?

In [ ]:
# Filtered variable: mu_{t|t} = E(mu_t | y_1,...,y_t), variance P_{t|t}.
filtered_mu  = pd.Series(res_ll.filter_results.filtered_state[0], index=y.index)
filtered_se  = np.sqrt(pd.Series(res_ll.filter_results.filtered_state_cov[0, 0], index=y.index))
f_lo, f_hi = filtered_mu - 1.96 * filtered_se, filtered_mu + 1.96 * filtered_se

print("Filtered variable:  mu_hat_{t|t} = E(mu_t | y_1,...,y_t);  95% CI = mu_{t|t} +/- 1.96*sqrt(P_{t|t})")
print(f"Mean filtered standard error (t>=2): {filtered_se.iloc[1:].mean():.4f}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y.index, y, color="gray", linewidth=0.9, alpha=0.6, label="Observed volatility")
ax.plot(filtered_mu.index, filtered_mu, color="#1f77b4", linewidth=1.8, label=r"Filtered $\hat{\mu}_{t|t}$")
ax.fill_between(filtered_mu.index, f_lo, f_hi, color="#1f77b4", alpha=0.18, label="Pointwise 95% CI")
ax.set_title("Filtered Local Level with Pointwise 95% Confidence Interval")
ax.set_xlabel("Observation index t"); ax.set_ylabel("Realized volatility"); ax.legend(loc="upper right")
plt.show()

**(c)** Obtain time plots for the smoothed variables with pointwise 95% confidence interval.
What is the mathematical expression of these "smoothed variables"?

In [ ]:
# Smoothed variable: mu_{t|T} = E(mu_t | y_1,...,y_T), variance P_{t|T}.
smoothed_mu = pd.Series(res_ll.smoothed_state[0], index=y.index)
smoothed_se = np.sqrt(pd.Series(res_ll.smoothed_state_cov[0, 0], index=y.index))
s_lo, s_hi = smoothed_mu - 1.96 * smoothed_se, smoothed_mu + 1.96 * smoothed_se

print("Smoothed variable: mu_hat_{t|T} = E(mu_t | y_1,...,y_T);  95% CI = mu_{t|T} +/- 1.96*sqrt(P_{t|T})")
print(f"Mean smoothed standard error: {smoothed_se.mean():.4f}  (narrower than filtered: "
      f"{smoothed_se.mean() < filtered_se.iloc[1:].mean()})")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y.index, y, color="gray", linewidth=0.9, alpha=0.6, label="Observed volatility")
ax.plot(smoothed_mu.index, smoothed_mu, color="#d62728", linewidth=1.8, label=r"Smoothed $\hat{\mu}_{t|T}$")
ax.fill_between(smoothed_mu.index, s_lo, s_hi, color="#d62728", alpha=0.18, label="Pointwise 95% CI")
ax.set_title("Smoothed Local Level with Pointwise 95% Confidence Interval")
ax.set_xlabel("Observation index t"); ax.set_ylabel("Realized volatility"); ax.legend(loc="upper right")
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y.index, y, color="gray", linewidth=0.8, alpha=0.45, label="Observed volatility")
ax.plot(filtered_mu.index, filtered_mu, color="#1f77b4", linewidth=1.5, label=r"Filtered $\hat{\mu}_{t|t}$")
ax.plot(smoothed_mu.index, smoothed_mu, color="#d62728", linewidth=1.5, label=r"Smoothed $\hat{\mu}_{t|T}$")
ax.set_title("Filtered vs. Smoothed Estimates of the Latent Volatility Level")
ax.set_xlabel("Observation index t"); ax.set_ylabel("Realized volatility"); ax.legend(loc="upper right")
plt.show()

# Model adequacy: standardized Kalman innovations.
std_resid = pd.Series(res_ll.standardized_forecasts_error[0], index=y.index
                      ).replace([np.inf, -np.inf], np.nan).dropna()
lb = acorr_ljungbox(std_resid, lags=[10, 20], return_df=True)
print("Ljung-Box on standardized Kalman innovations:")
print(lb)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_acf(std_resid, ax=axes[0], lags=24, title="ACF of standardized innovations")
axes[1].hist(std_resid, bins=30, color="#4C78A8", alpha=0.8, density=True); axes[1].set_title("Histogram")
scistats.probplot(std_resid, dist="norm", plot=axes[2]); axes[2].set_title("QQ plot vs Normal")
plt.tight_layout(); plt.show()

## Problem 2. Gibbs Sampling and Markov Switching Model

For this problem, use the dataset `data/ge_returns.txt`. The dataset consists GE stock monthly log return from 1926 to 1999. We would
like to fit a Markov switching model as stated in Week 12, page 63, equation (12.55) and (12.56):

$$r_t=\beta_{s_t}\sqrt{h_t}+\sqrt{h_t}\,\epsilon_t,\quad
h_t=\alpha_{s_t,0}+\alpha_{s_t,1}h_{t-1}+\alpha_{s_t,2}a_{t-1}^2,\quad a_t=r_t-\beta_{s_t}\sqrt{h_t}.$$

The two-state model is estimated in the **Haas–Mittnik–Paolella (2004)** form (each regime has its
own path-independent GARCH variance $h_{k,t}$), which makes exact HMM filtering, exact FFBS Gibbs and
exact Baum–Welch possible. State 2 is the higher risk-premium regime ($\beta_2>\beta_1$).

**(a)** Do a Gibbs sampling to obtain the posterior distribution as in Figure 12.16, page
69-70. You can choose your own prior parameters. Pay special notice how you sample $\alpha_{ij}$, as
stated in page 66.

In [ ]:
r = pd.read_csv("data/ge_returns.txt", header=None, sep=r"\s+")[0].astype(float)
dates = pd.date_range("1926-01-01", periods=len(r), freq="MS")
r = pd.Series(r.to_numpy(), index=dates, name="GE monthly log return")
r_values = r.to_numpy()
print(f"GE monthly log returns: {len(r)} obs, {r.index[0].date()} to {r.index[-1].date()}")

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(r.index, r, color="black", linewidth=0.9); ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("GE Monthly Log Returns, 1926-1999")
ax.set_xlabel("Date"); ax.set_ylabel("Monthly log return")
plt.show()

In [ ]:
# ---- constraints / transformation for alpha (alpha_i0>0, alpha_i1,alpha_i2>=0, sum<1) ----
def transition_matrix(e1, e2):
    return np.array([[1.0 - e1, e1], [e2, 1.0 - e2]], dtype=float)

def alpha_from_u(u, cap=0.995):
    u = np.asarray(u, float); e = np.exp(np.clip(u[1:], -40, 40)); d = 1.0 + e[0] + e[1]
    return np.array([np.exp(np.clip(u[0], -40, 40)), cap * e[0] / d, cap * e[1] / d])

def log_alpha_jacobian(u, cap=0.995):
    u = np.asarray(u, float); e1, e2 = np.exp(np.clip(u[1:], -40, 40)); d = 1.0 + e1 + e2
    return u[0] + 2*np.log(cap) + np.log(e1/d) + np.log(e2/d) + np.log(1.0/d)

def alpha_to_u(alpha, cap=0.995):
    alpha = np.asarray(alpha, float); slack = max(cap - alpha[1] - alpha[2], 1e-8)
    return np.array([np.log(max(alpha[0], 1e-8)), np.log(max(alpha[1], 1e-8)/slack),
                     np.log(max(alpha[2], 1e-8)/slack)])

def check_alpha(alpha):
    alpha = np.asarray(alpha, float)
    return bool(np.all(np.isfinite(alpha)) and np.all(alpha[:, 0] > 0)
                and np.all(alpha[:, 1:] >= 0) and np.all(alpha[:, 1] + alpha[:, 2] < 1))

def garch_paths(r_values, beta, alpha):
    # Haas per-regime PATH-INDEPENDENT variance + emission log-densities (no beta-order check;
    # labeling is handled separately).
    rv = np.asarray(r_values, float); beta = np.asarray(beta, float); alpha = np.asarray(alpha, float)
    if not check_alpha(alpha):
        return None, None
    T = rv.shape[0]; v = float(np.var(rv)); h = np.empty((T, 2)); logf = np.empty((T, 2))
    for k in (0, 1):
        a0, a1, a2 = alpha[k]; bk = beta[k]; h_prev = v
        for t in range(T):
            if t == 0:
                h_t = v
            else:
                a_prev = rv[t-1] - bk*math.sqrt(h_prev); h_t = a0 + a1*h_prev + a2*a_prev*a_prev
            if h_t <= 1e-10 or not math.isfinite(h_t):
                return None, None
            res = rv[t] - bk*math.sqrt(h_t)
            logf[t, k] = -0.5*(LOG2PI + math.log(h_t) + res*res/h_t); h[t, k] = h_t; h_prev = h_t
    return h, logf

def stationary_dist(e1, e2):
    s = e1 + e2
    return np.array([0.5, 0.5]) if s <= 0 else np.array([e2/s, e1/s])

def forward_filter(logf, e1, e2):
    T = logf.shape[0]; P = transition_matrix(e1, e2)
    p00, p01, p10, p11 = P[0,0], P[0,1], P[1,0], P[1,1]
    pi = stationary_dist(e1, e2); filtered = np.empty((T, 2)); loglik = 0.0; f0, f1 = pi
    for t in range(T):
        pr0, pr1 = (pi[0], pi[1]) if t == 0 else (f0*p00 + f1*p10, f0*p01 + f1*p11)
        u0 = pr0*math.exp(logf[t,0]); u1 = pr1*math.exp(logf[t,1]); sc = u0 + u1
        if sc <= 0 or not math.isfinite(sc):
            sc = 1e-300; f0 = f1 = 0.5
        else:
            f0, f1 = u0/sc, u1/sc
        loglik += math.log(sc); filtered[t, 0] = f0; filtered[t, 1] = f1
    return filtered, loglik, P

def ffbs_sample(filtered, P, rng):
    T = filtered.shape[0]; states = np.zeros(T, dtype=int)
    states[-1] = rng.choice(2, p=filtered[-1])
    for t in range(T-2, -1, -1):
        w = filtered[t]*P[:, states[t+1]]; s = w.sum()
        states[t] = rng.choice(2, p=(w/s if s > 0 else np.array([0.5, 0.5])))
    return states

def forward_backward(logf, e1, e2):
    filtered, loglik, P = forward_filter(logf, e1, e2); T = filtered.shape[0]; emit = np.exp(logf)
    bmsg = np.ones((T, 2))
    for t in range(T-2, -1, -1):
        bmsg[t] = P @ (emit[t+1]*bmsg[t+1]); bmsg[t] /= max(bmsg[t].sum(), 1e-300)
    gamma = filtered*bmsg; gamma /= gamma.sum(axis=1, keepdims=True)
    xi = np.zeros((T-1, 2, 2))
    for t in range(T-1):
        nm = filtered[t][:, None]*P*(emit[t+1]*bmsg[t+1])[None, :]; xi[t] = nm/max(nm.sum(), 1e-300)
    return gamma, xi, loglik

def transition_counts(states):
    st = np.asarray(states, int)
    return (int(np.sum((st[:-1]==0)&(st[1:]==0))), int(np.sum((st[:-1]==0)&(st[1:]==1))),
            int(np.sum((st[:-1]==1)&(st[1:]==0))), int(np.sum((st[:-1]==1)&(st[1:]==1))))

def complete_loglik(r_values, states, beta, alpha):
    h, logf = garch_paths(r_values, beta, alpha)
    if logf is None:
        return -np.inf
    st = np.asarray(states, int)
    return float(logf[np.arange(len(st)), st].sum())
print("core functions defined")

In [ ]:
def run_gibbs(r_values, n_iter=12000, burn=4000, thin=4, seed=123, b0=0.0, B0=100.0, verbose=True):
    # Data-augmentation Gibbs:
    #   states S | theta  -> exact FFBS
    #   e1,e2 | S         -> conjugate Beta
    #   beta  | S, h      -> CONJUGATE NORMAL (closed form; B's technique): r_t/sqrt(h_{s_t,t}) ~ N(beta_{s_t},1)
    #   alpha | S, beta   -> random-walk Metropolis on the constrained transform (flat alpha prior + Jacobian)
    # Label identification: relabel each sweep so beta2 > beta1.
    rng = np.random.default_rng(seed)
    states = (r_values > np.median(r_values)).astype(int)
    beta = np.array([-0.05, 0.10]); v = np.var(r_values)
    u_alpha = np.r_[alpha_to_u([0.05*v, 0.85, 0.08]), alpha_to_u([0.08*v, 0.80, 0.12])]
    e1, e2 = 0.04, 0.08
    sd_alpha = np.array([0.08, 0.06, 0.06, 0.08, 0.06, 0.06])
    saved_theta, saved_e, saved_states = [], [], []
    acc_alpha = 0; t0 = time.time()
    for it in range(1, n_iter + 1):
        alpha = np.vstack([alpha_from_u(u_alpha[:3]), alpha_from_u(u_alpha[3:])])
        h, logf = garch_paths(r_values, beta, alpha)
        filtered, _, P = forward_filter(logf, e1, e2)
        states = ffbs_sample(filtered, P, rng)                       # exact state draw

        n11, n12, n21, n22 = transition_counts(states)
        e1 = rng.beta(2 + n12, 20 + n11); e2 = rng.beta(2 + n21, 20 + n22)

        sq = np.sqrt(h)                                              # conjugate beta (given h)
        for k in (0, 1):
            idx = states == k; nk = int(idx.sum())
            if nk > 0:
                xk = r_values[idx] / sq[idx, k]
                prec = 1.0/B0 + nk; mean = (b0/B0 + xk.sum())/prec
                beta[k] = rng.normal(mean, math.sqrt(1.0/prec))
            else:
                beta[k] = rng.normal(b0, math.sqrt(B0))

        prop = u_alpha + rng.normal(0, sd_alpha)                     # alpha RW-MH
        a_prop = np.vstack([alpha_from_u(prop[:3]), alpha_from_u(prop[3:])])
        lp_prop = complete_loglik(r_values, states, beta, a_prop) + log_alpha_jacobian(prop[:3]) + log_alpha_jacobian(prop[3:])
        lp_cur  = complete_loglik(r_values, states, beta, alpha)     + log_alpha_jacobian(u_alpha[:3]) + log_alpha_jacobian(u_alpha[3:])
        if np.log(rng.uniform()) < lp_prop - lp_cur:
            u_alpha = prop; acc_alpha += 1

        if beta[0] > beta[1]:                                        # relabel: state 2 = higher beta
            beta = beta[::-1].copy()
            u_alpha = np.r_[u_alpha[3:], u_alpha[:3]].copy()
            e1, e2 = e2, e1; states = 1 - states

        if it > burn and (it - burn) % thin == 0:
            al = np.vstack([alpha_from_u(u_alpha[:3]), alpha_from_u(u_alpha[3:])])
            saved_theta.append(np.r_[beta, al.ravel()]); saved_e.append([e1, e2]); saved_states.append(states.copy())
        if verbose and it % max(n_iter // 10, 1) == 0:
            print(f"iteration {it:>6}/{n_iter}, saved={len(saved_theta)}, elapsed={time.time()-t0:5.1f}s")

    cols = ["beta1", "beta2", "alpha10", "alpha11", "alpha12", "alpha20", "alpha21", "alpha22"]
    draws = pd.DataFrame(saved_theta, columns=cols); draws[["e1", "e2"]] = np.asarray(saved_e)
    accept = {"alpha_accept_rate": acc_alpha/n_iter, "beta_update": "conjugate Normal (closed form)",
              "state_update": "exact FFBS Gibbs", "n_iter": n_iter, "burn": burn, "thin": thin,
              "n_draws": len(draws)}
    return draws, np.asarray(saved_states), accept

FAST_DEMO, FORCE_RERUN = False, False
CACHE_DRAWS, CACHE_STATES, CACHE_ACCEPT = ".cache/p2_mcmc_draws.csv", ".cache/p2_state_draws.npy", ".cache/p2_accept.json"

if FAST_DEMO:
    mcmc_draws, state_draws, accept_rates = run_gibbs(r_values, 200, 80, 4, RNG_SEED); RESULT_MODE = "DEMO"
else:
    os.makedirs(".cache", exist_ok=True)
    if all(os.path.exists(f) for f in (CACHE_DRAWS, CACHE_STATES, CACHE_ACCEPT)) and not FORCE_RERUN:
        print(">>> FINAL: loading cached draws.")
        mcmc_draws = pd.read_csv(CACHE_DRAWS); state_draws = np.load(CACHE_STATES)
        accept_rates = json.load(open(CACHE_ACCEPT))
    else:
        print(">>> FINAL: running full MCMC (12000/4000/4); a few minutes.")
        mcmc_draws, state_draws, accept_rates = run_gibbs(r_values, 12000, 4000, 4, RNG_SEED)
        mcmc_draws.to_csv(CACHE_DRAWS, index=False); np.save(CACHE_STATES, state_draws)
        json.dump(accept_rates, open(CACHE_ACCEPT, "w"), indent=2)
    RESULT_MODE = "FINAL"
print("Mode:", RESULT_MODE, "| sampler:", accept_rates)

In [ ]:
param_cols = ["beta1", "beta2", "alpha10", "alpha11", "alpha12", "alpha20", "alpha21", "alpha22", "e1", "e2"]
summ = mcmc_draws[param_cols].agg(["mean", "std"]).T
summ["2.5%"] = mcmc_draws[param_cols].quantile(0.025); summ["97.5%"] = mcmc_draws[param_cols].quantile(0.975)
print(f"Posterior summary ({RESULT_MODE}, {len(mcmc_draws)} draws):")
display(summ.round(4))
print(f"alpha RW-MH acceptance = {accept_rates['alpha_accept_rate']:.3f}; "
      f"beta update = {accept_rates['beta_update']}; states = {accept_rates['state_update']}")

# Figure A: the four headline parameters (risk premium + transition probabilities).
key = [("beta1", r"$\beta_1$ (State 1, low-beta)"), ("beta2", r"$\beta_2$ (State 2, high-beta)"),
       ("e1", r"$e_1=P(s_t=2\mid s_{t-1}=1)$"), ("e2", r"$e_2=P(s_t=1\mid s_{t-1}=2)$")]
fig, axes = plt.subplots(2, 2, figsize=(11, 7)); axes = axes.ravel()
for ax, (c, t) in zip(axes, key):
    ax.hist(mcmc_draws[c], bins=40, density=True, color="#4C78A8", alpha=0.85)
    ax.axvline(mcmc_draws[c].mean(), color="#D62728", lw=1.2)
    ax.set_title("Posterior of " + t); ax.set_xlabel(c)
fig.suptitle("Posterior Distributions: Risk Premium and Transition Probabilities", y=1.0)
plt.tight_layout(); plt.show()

# Figure B: the six GARCH parameters.
garch_cols = ["alpha10", "alpha11", "alpha12", "alpha20", "alpha21", "alpha22"]
fig, axes = plt.subplots(2, 3, figsize=(13, 7)); axes = axes.ravel()
for ax, c in zip(axes, garch_cols):
    ax.hist(mcmc_draws[c], bins=40, density=True, color="#4C78A8", alpha=0.85)
    ax.axvline(mcmc_draws[c].mean(), color="#D62728", lw=1.2); ax.set_title(c)
fig.suptitle("Posterior Distributions: GARCH Parameters", y=1.0)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(5, 2, figsize=(12, 14)); axes = axes.ravel()
for ax, c in zip(axes, param_cols):
    ax.plot(mcmc_draws[c].to_numpy(), color="#4C78A8", lw=0.7); ax.set_title(f"trace: {c}")
fig.suptitle(f"MCMC Trace Plots ({RESULT_MODE})", y=1.005); plt.tight_layout(); plt.show()

ok = ((mcmc_draws[["alpha10","alpha20"]] > 0).all().all()
      and (mcmc_draws[["alpha11","alpha12","alpha21","alpha22"]] >= 0).all().all()
      and ((mcmc_draws["alpha11"]+mcmc_draws["alpha12"]) < 1).all()
      and ((mcmc_draws["alpha21"]+mcmc_draws["alpha22"]) < 1).all()
      and (mcmc_draws["beta2"] >= mcmc_draws["beta1"]).all())
print("All draws satisfy alpha constraints and beta2>=beta1:", bool(ok))

**(b)** Based on the algorithm in (a), present the posterior distribution of state 2 as in the
bottom figure on page 68.

In [ ]:
prob_state2_gibbs = pd.Series(state_draws.mean(axis=0), index=r.index, name="Gibbs P(state 2)")
print(f"P(s_t=2 | r): mean = {prob_state2_gibbs.mean():.3f}; share of months with P>0.5 = "
      f"{(prob_state2_gibbs > 0.5).mean():.3f}")

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(prob_state2_gibbs.index, prob_state2_gibbs, color="#D62728", linewidth=1.3)
ax.fill_between(prob_state2_gibbs.index, 0, prob_state2_gibbs, color="#D62728", alpha=0.18)
ax.set_ylim(0, 1); ax.set_title("Posterior Probability of State 2 (Gibbs)")
ax.set_xlabel("Date"); ax.set_ylabel(r"$P(s_t=2 \mid r)$"); plt.show()

**(c)** Instead of the Bayesian way in (a)–(b), we can also do it through a Baum-Welch
algorithm. Do it and present the estimation for the parameters.

In [ ]:
# Exact Baum-Welch for the Haas MS-GARCH (path-independent emissions).
def params_from_free(theta):
    b1 = theta[0]; b2 = b1 + np.exp(np.clip(theta[1], -40, 40))
    return np.array([b1, b2]), np.vstack([alpha_from_u(theta[2:5]), alpha_from_u(theta[5:8])])

def free_from_params(beta, alpha):
    return np.r_[beta[0], np.log(max(beta[1]-beta[0], 1e-8)), alpha_to_u(alpha[0]), alpha_to_u(alpha[1])]

def neg_Q(theta, r_values, gamma):
    beta, alpha = params_from_free(theta); h, logf = garch_paths(r_values, beta, alpha)
    return 1e100 if logf is None else float(-np.sum(gamma*logf))

def run_baum_welch(r_values, start_draws, max_iter=300, tol=1e-6, verbose=True):
    s = start_draws.mean()
    beta = np.array([s["beta1"], s["beta2"]])
    alpha = np.array([[s["alpha10"], s["alpha11"], s["alpha12"]], [s["alpha20"], s["alpha21"], s["alpha22"]]])
    e1, e2 = float(s["e1"]), float(s["e2"]); theta = free_from_params(beta, alpha); ll_trace = []
    for k in range(1, max_iter + 1):
        beta, alpha = params_from_free(theta); h, logf = garch_paths(r_values, beta, alpha)
        gamma, xi, ll = forward_backward(logf, e1, e2); ll_trace.append(ll)
        tc = xi.sum(axis=0)
        e1 = float(np.clip(tc[0,1]/max(tc[0].sum(), 1e-12), 1e-4, 0.999))
        e2 = float(np.clip(tc[1,0]/max(tc[1].sum(), 1e-12), 1e-4, 0.999))
        cur = neg_Q(theta, r_values, gamma)
        opt = minimize(neg_Q, theta, args=(r_values, gamma), method="Nelder-Mead",
                       options={"maxiter": 800, "xatol": 1e-6, "fatol": 1e-6})
        if np.isfinite(opt.fun) and opt.fun <= cur:
            theta = opt.x
        if k > 1 and abs(ll_trace[-1] - ll_trace[-2]) < tol:
            if verbose: print(f"EM converged at iteration {k}.")
            break
    beta, alpha = params_from_free(theta); h, logf = garch_paths(r_values, beta, alpha)
    gamma, xi, ll = forward_backward(logf, e1, e2)
    est = pd.Series({"beta1": beta[0], "beta2": beta[1], "alpha10": alpha[0,0], "alpha11": alpha[0,1],
                     "alpha12": alpha[0,2], "alpha20": alpha[1,0], "alpha21": alpha[1,1],
                     "alpha22": alpha[1,2], "e1": e1, "e2": e2, "loglik": ll, "iterations": len(ll_trace)})
    return est, gamma, np.asarray(ll_trace)

em_estimates, em_gamma, em_ll_trace = run_baum_welch(r_values, mcmc_draws, max_iter=300)
print("Exact Baum-Welch / EM estimates:")
display(em_estimates.to_frame("estimate").round(4))
_dmin = float(np.diff(em_ll_trace).min())
print(f"EM log-likelihood non-decreasing up to numerical tolerance: {_dmin > -1e-3} "
      f"(largest step decrease {_dmin:.2e})")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(np.arange(1, len(em_ll_trace)+1), em_ll_trace, marker="o", linewidth=1.2)
ax.set_title("Exact Baum-Welch / EM Log-Likelihood Trace")
ax.set_xlabel("EM iteration"); ax.set_ylabel("HMM log-likelihood"); plt.show()

**(d)** Based on the algorithm in (b), present the inferred states for each time point.

In [ ]:
prob_state2_em = pd.Series(em_gamma[:, 1], index=r.index, name="EM P(state 2)")
gibbs_state = (prob_state2_gibbs > 0.5).astype(int) + 1
em_state    = (prob_state2_em    > 0.5).astype(int) + 1
state_table = pd.DataFrame({"return": r, "gibbs_prob_state2": prob_state2_gibbs,
                            "gibbs_inferred_state": gibbs_state, "em_prob_state2": prob_state2_em,
                            "em_inferred_state": em_state})
print(f"Gibbs/EM inferred-state agreement: {(gibbs_state.values == em_state.values).mean():.3f}")
display(state_table.head(10))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(r.index, r, color="black", linewidth=0.8, label="GE return")
# State 2 is dominant (~95%), so shade the RARE regime (state 1: low/negative-beta, high-volatility).
ax.fill_between(r.index, 0, 1, where=(gibbs_state.values == 1), transform=ax.get_xaxis_transform(),
                color="#D62728", alpha=0.18, label="Gibbs state 1 (rare, high-volatility)")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("GE Returns with Gibbs-Inferred Rare State (State 1) Highlighted")
ax.set_xlabel("Date"); ax.set_ylabel("Monthly log return"); ax.legend(loc="upper right"); plt.show()

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(prob_state2_gibbs.index, prob_state2_gibbs, label="Gibbs P(state 2)", color="#D62728", lw=1.3)
ax.plot(prob_state2_em.index, prob_state2_em, label="EM smoothed P(state 2)", color="#1F77B4", lw=1.3, alpha=0.85)
ax.set_ylim(0, 1); ax.set_title("State 2 Probability: Gibbs vs. Exact Baum-Welch / EM")
ax.set_xlabel("Date"); ax.set_ylabel("Probability"); ax.legend(loc="upper right"); plt.show()

## Problem 3. VAR model and Cointegration

For this problem, use the dataset `data/us_interest_rates.txt`. The dataset consists the monthly U.S.
1-year $r_{1t}$ (first column) and 3-year $r_{3t}$ (second column) Treasury constant maturity rates
from April 1953 to March 2004.

**(a)** Fit a VAR model on the two dimensional data $(r_{1t}, r_{3t})$.

In [ ]:
raw = pd.read_csv("data/us_interest_rates.txt", header=None, sep=r"\s+")
df = pd.DataFrame({"r1": raw.iloc[:, 0].astype(float), "r3": raw.iloc[:, 1].astype(float),
                   "date": pd.to_datetime({"year": raw.iloc[:, 2].astype(int),
                                           "month": raw.iloc[:, 3].astype(int),
                                           "day": raw.iloc[:, 4].astype(int)})}).set_index("date")
print(f"Shape {df.shape}; {df.index[0].date()} to {df.index[-1].date()}")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df["r1"], label=r"1-year $r_{1t}$", lw=1.2)
ax.plot(df.index, df["r3"], label=r"3-year $r_{3t}$", lw=1.2)
ax.set_title("U.S. Treasury Constant Maturity Rates"); ax.set_xlabel("Date"); ax.set_ylabel("Rate (%)")
ax.legend(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(df["r1"], ax=axes[0,0], lags=24, title="ACF: r1 level (slow decay)")
plot_acf(df["r3"], ax=axes[0,1], lags=24, title="ACF: r3 level (slow decay)")
plot_acf(df["r1"].diff().dropna(), ax=axes[1,0], lags=24, title="ACF: diff r1")
plot_acf(df["r3"].diff().dropna(), ax=axes[1,1], lags=24, title="ACF: diff r3")
plt.tight_layout(); plt.show()

In [ ]:
model = VAR(df[["r1", "r3"]])
sel = model.select_order(maxlags=12)
print(sel.summary())
print("Argmin of each criterion:", sel.selected_orders)

p_aic = int(sel.selected_orders["aic"])   # 12
p_bic = int(sel.selected_orders["bic"])   # 3
print(f"AIC selects VAR({p_aic}); BIC selects VAR({p_bic}).")

var_aic = model.fit(p_aic)
var_bic = model.fit(p_bic)
print(f"\nVAR({p_aic}) stable: {var_aic.is_stable()};  VAR({p_bic}) stable: {var_bic.is_stable()}")

# Write out the BIC-selected VAR(3) equations explicitly (B's style).
print(f"\nEstimated VAR({p_bic}) equations (BIC model):")
prm = var_bic.params
for eq in ["r1", "r3"]:
    terms = " + ".join(f"{prm.loc[idx, eq]:+.4f}*{idx}" for idx in prm.index)
    print(f"  {eq}_t = {terms}")
print(f"\nResidual correlation corr(u1,u3) = {np.corrcoef(var_bic.resid['r1'], var_bic.resid['r3'])[0,1]:.3f}")

**(b)** How do you check the adequacy of your fitted VAR model? Perform such check under 5%
significance level.

In [ ]:
# Multivariate Portmanteau residual whiteness test (H0: residuals are white noise).
for lab, res_v, pp in [("AIC VAR(%d)" % p_aic, var_aic, p_aic), ("BIC VAR(%d)" % p_bic, var_bic, p_bic)]:
    wt = res_v.test_whiteness(nlags=pp + 6, adjusted=True)
    print(f"{lab}: Portmanteau p-value = {wt.pvalue:.3e} -> "
          f"{'adequate' if wt.pvalue > 0.05 else 'NOT adequate (reject white noise)'} at 5%")

# Honest lag scan: which levels-VAR(p) passes the whiteness test at 5%?
rows = []
for pp in range(1, 25):
    rp = model.fit(pp); pv = rp.test_whiteness(nlags=pp + 6, adjusted=True).pvalue
    rows.append({"p": pp, "whiteness_p": pv, "white@5%": pv > 0.05})
scan = pd.DataFrame(rows)
display(scan.round(4))
print("Lag orders with white residuals at 5%:", scan.loc[scan["white@5%"], "p"].tolist() or "NONE")

resid = var_aic.resid
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(resid.index, resid["r1"], color="#1f77b4", lw=1.0); axes[0].axhline(0, color="gray", lw=.8)
axes[0].set_title(f"VAR({p_aic}) residuals: r1")
axes[1].plot(resid.index, resid["r3"], color="#d62728", lw=1.0); axes[1].axhline(0, color="gray", lw=.8)
axes[1].set_title(f"VAR({p_aic}) residuals: r3"); axes[1].set_xlabel("Date")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(resid["r1"], ax=axes[0], lags=24, title="ACF of r1 residuals")
plot_acf(resid["r3"], ax=axes[1], lags=24, title="ACF of r3 residuals")
plt.tight_layout(); plt.show()

print("\nConclusion: under AIC the levels VAR(12) residuals REJECT whiteness (p<<0.05) -> not adequate;")
print("the BIC VAR(3) is even less adequate. Only very long lags (p=22-24) whiten the residuals.")

**(c)** We want to know whether the two interest rate series are cointegrated or not. What
does "cointegrated" mean mathematically? What kind of test can you test this? Use 5% significance
level to perform the test.

In [ ]:
# Cointegration: r1t, r3t are I(1) but a linear combination beta'y_t is I(0).
def adf_row(s, name):
    st, pv, lag, n, crit, _ = adfuller(s.dropna(), autolag="AIC")
    return {"series": name, "ADF stat": st, "p-value": pv, "5% crit": crit["5%"], "stationary@5%": pv < 0.05}

adf = pd.DataFrame([adf_row(df["r1"], "r1 level"), adf_row(df["r3"], "r3 level"),
                    adf_row(df["r1"].diff(), "diff r1"), adf_row(df["r3"].diff(), "diff r3"),
                    adf_row(df["r3"] - df["r1"], "spread r3-r1")])
display(adf.round(4))
print("=> r1, r3 are I(1) (unit root in levels, stationary in differences).")

# Test 1: Johansen trace test (det_order=0, k_ar_diff = max(p-1,0)).
k_ar_diff = max(p_aic - 1, 0)
joh = coint_johansen(df[["r1", "r3"]], det_order=0, k_ar_diff=k_ar_diff)
jt = pd.DataFrame({"H0 rank<=r": [0, 1], "trace stat": joh.lr1, "5% crit": joh.cvt[:, 1],
                   "reject@5%": joh.lr1 > joh.cvt[:, 1]})
display(jt.round(4))
rank = int(np.sum(joh.lr1 > joh.cvt[:, 1]))
print(f"Johansen (det_order=0, k_ar_diff={k_ar_diff}) rank at 5% = {rank}")

# Sensitivity of the rank to deterministic term and lag.
sens = []
for det in (-1, 0, 1):
    for kad in sorted({max(p_aic-1, 0), 1, 2, 11}):
        jj = coint_johansen(df[["r1", "r3"]], det_order=det, k_ar_diff=kad)
        sens.append({"det_order": det, "k_ar_diff": kad,
                     "rank@5%": int(np.sum(jj.lr1 > jj.cvt[:, 1]))})
display(pd.DataFrame(sens))

# Test 2: Engle-Granger two-step cointegration test (B's technique).
eg_t, eg_p, eg_crit = coint(df["r3"], df["r1"], trend="c", autolag="AIC")
print(f"\nEngle-Granger test (regress r3 on r1): t-stat = {eg_t:.4f}, p-value = {eg_p:.4f}, "
      f"5% crit = {eg_crit[1]:.4f} -> {'cointegrated' if eg_p < 0.05 else 'no cointegration'} at 5%")

# Cointegrating vector / spread.
bv = joh.evec[:, 0]; nv = bv / bv[1]; gamma = -nv[0]
spread = df["r3"] - gamma * df["r1"]
print(f"Johansen cointegrating vector (normalized r3=1): r3 - ({gamma:.4f})*r1  (approx r3 - r1)")
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(spread.index, spread, color="#2ca02c", lw=1.2)
ax.axhline(spread.mean(), color="gray", ls="--", lw=1.0, label="sample mean")
ax.set_title("Estimated Cointegrating Spread  r3 - gamma*r1"); ax.set_xlabel("Date"); ax.legend(); plt.show()

print("\nConclusion (5%): r1, r3 are I(1); both the Engle-Granger test and Johansen (at parsimonious")
print("lags / no-constant, and the stationary spread) support ONE cointegrating relation, vector ~ (1,-1),")
print("i.e. the term spread r3-r1 is stationary. The lone rank=2 at long lag + constant is not robust.")